# Projeto de Redes Neurais: quanto cada arquitetura depende do histórico?

**Pergunta.** Redes de previsão de séries temporais costumam ser comparadas só pelo erro. Este projeto mede **como** cada arquitetura usa o passado: se ela depende da ordem do histórico, se usa o passado distante e quanto ganha além de simplesmente copiar o último valor ou o último dia.

**Modelos (5 famílias):** DLinear (linear) · LSTM (recorrente) · TCN (convolucional) · Informer e PatchTST (Transformers).
**Bases:** ETTh1 (horária) e Weather (a cada 10 min).
**Testes:** (1) embaralhar a janela de entrada · (2) variar o tamanho da janela · (3) comparar com baselines simples.

## Estrutura do projeto

| Item | Conteúdo |
|---|---|
| `00_LEIA-ME` | este guia: pergunta, protocolo e decisões |
| `01_dados` | download, inspeção, limpeza, split e normalização das bases → `data/processed/` |
| `02_codigo_comum` | código compartilhado (dados, perturbações, treino e avaliação); gera `src/rn_utils.py` |
| `03_baselines` | naive, seasonal naive e média da janela |
| `10_dlinear` a `14_patchtst` | um notebook por modelo: arquitetura, treino e resultados |
| `90_analise` | tabelas, figuras e testes estatísticos do artigo |
| `src/rn_utils.py` | gerado pelo `02_codigo_comum` (não editar à mão) |
| `data/raw`, `data/processed` | bases originais e bases tratadas |
| `resultados/<base>/<modelo>/` | por execução: `.json` (métricas) e `.npz` (erro de cada amostra de teste) |
| `checkpoints/` | pesos dos modelos treinados |
| `figuras/` | figuras e tabelas geradas |

## Como reproduzir

Ordem: `01_dados` → `02_codigo_comum` → `03_baselines` → `10` a `14` → `90_analise`.

- Os resultados de todas as execuções já estão em `resultados/`. Rodar os notebooks de novo apenas recarrega e lista esses resultados, sem treinar. Para isso basta o ambiente padrão (CPU).
- Para treinar do zero, apague `resultados/` e `checkpoints/` e use GPU (**Ambiente de execução → Alterar o tipo → GPU T4**). Uma execução interrompida continua da última época salva quando a célula é rodada de novo.
- Os notebooks esperam a pasta em `/content/drive/MyDrive/RN-Projeto`. Se ela foi compartilhada com você, adicione um atalho no Meu Drive: botão direito na pasta → **Organizar → Adicionar atalho → Meu Drive**.

## Protocolo experimental

| Decisão | Escolha | Por quê |
|---|---|---|
| Variáveis | **Univariado**: prever só a variável `OT` a partir do passado de `OT` | DLinear e PatchTST tratam cada variável separadamente; Informer, LSTM e TCN misturam variáveis. No modo univariado essa diferença desaparece e a comparação mede só o uso do **tempo**. Também deixa o Weather barato de rodar. |
| Covariáveis de calendário | **Nenhuma** (sem hora do dia ou dia da semana) | Todos recebem a mesma informação. E, se o modelo recebesse a hora de cada passo, embaralhar os valores ainda deixaria a ordem recuperável pelos carimbos de tempo, o que invalidaria o teste 1. |
| Split ETTh1 | 12 / 4 / 4 meses (8640 / 2880 / 2880 horas) | Padrão da literatura (Informer, DLinear, PatchTST). |
| Split Weather | 70% / 10% / 20%, na ordem do tempo | Padrão da literatura. |
| Normalização global | z-score com média e desvio **só do treino** | Evita vazamento de informação do teste. Métricas são reportadas nessa escala, como nos artigos. |
| Normalização por janela | **RevIN em todos os modelos** (wrapper `ComRevIN`) | O PatchTST original usa RevIN e os outros não; aplicar a todos evita que essa vantagem seja confundida com a arquitetura. Média e desvio não mudam ao embaralhar, então isso não vaza informação de ordem. |
| Horizonte | H = 96 | Núcleo do projeto. |
| Janelas | ETTh1: 96, 192, 336, 512, 720 · Weather: 96, 336, 720 | Grade reduzida no Weather por custo. |
| Mesmos alvos para toda janela | Os pontos previstos no treino, na validação e no teste são **os mesmos para qualquer L**. O treino começa no passo 720. | Senão, comparar L = 96 com L = 720 compararia pontos diferentes. |
| Orçamento de treino | Adam, perda MSE, lote 64, até 30 épocas, early stopping com paciência 5 na validação, clip de gradiente 1.0 | Igual para todos. |
| Ajuste de hiperparâmetros | Cada modelo escolhe o learning rate entre {1e-3, 3e-4, 1e-4} pela validação (L = 336, seed 0), em cada base. Os outros hiperparâmetros seguem os artigos originais. | Mesmo orçamento de ajuste para todos. Sem isso, LSTM e TCN (sem configuração publicada para essas bases) sairiam prejudicados. |
| Seeds | 0, 1, 2 nas duas bases, para todos os modelos | Média ± desvio padrão. O plano inicial tinha 1 seed no Weather, por custo. Nas execuções preliminares, o erro de teste de um mesmo modelo e janela variou mais que as diferenças entre modelos só com a troca do learning rate, e o protocolo foi estendido para 3 seeds em **todos** os modelos, antes de qualquer comparação final. |
| Métricas | MSE e MAE (escala normalizada), MSE sem os dias suspeitos (ETTh1) e **ganho sobre os baselines** = 1 − MSE / MSE do baseline | O ganho responde quanto a rede acrescenta além de copiar o último valor ou o último dia. |
| Comparação final | Cada modelo na janela de **menor erro médio de validação**. Diferenças testadas com Diebold–Mariano (variância de Newey–West, correção de Holm) | A janela nunca é escolhida pelo teste. O teste estatístico leva em conta que previsões vizinhas se sobrepõem. |

### Testes de embaralhamento

Feitos **só na avaliação**, sobre o modelo já treinado, com 3 permutações diferentes por teste (reporta-se a média). As permutações dependem só de (base, janela, teste, repetição), então **todos os modelos veem exatamente as mesmas janelas embaralhadas**.

| Teste | O que faz | O que revela |
|---|---|---|
| `embaralhar_tudo` | embaralha os L passos | o modelo usa a ordem? |
| `embaralhar_antigo` | mantém os últimos 25% e embaralha o resto | o modelo usa a ordem do passado distante? |
| `embaralhar_recente` | embaralha só os últimos 25% | o modelo depende da ordem do passado recente (atalho tipo naive)? |
| `embaralhar_blocos` | troca a ordem de blocos de 16 passos, mantendo a ordem dentro de cada bloco | o modelo usa ordem local ou global? |

**Como interpretar:** se o erro **não** piora, a conclusão é segura: o modelo não usa aquela ordem. Se piora, o modelo é sensível à ordem, mas parte da piora pode vir de a janela embaralhada ser diferente de tudo que ele viu no treino. Isso entra como limitação no artigo.

### Baselines

`naive` (repete o último valor) · `sazonal` (repete o último dia: 24 passos no ETTh1, 144 no Weather) · `media_janela` (média da janela; é o que um modelo com RevIN prevê se não aprender nada).

## Problemas encontrados nos dados (ver `01_dados`)

**Weather**
- 50 valores sentinela `-9999` na variável alvo `OT` (CO₂ em ppm), em dois blocos (34 e 16 passos, ambos no treino). Em `wv` e `max. PAR`, que não usamos, também há `-9999`. **Se não fossem tratados, o desvio usado na normalização seria inflado em cerca de 20 vezes.**
- 1 linha duplicada (2020-05-12 06:00) e 1 buraco de 100 min (2020-05-29, 9 passos faltando).
- As leituras logo antes e logo depois de um dos blocos são 381,1 e 305,5 ppm, implausíveis para CO₂ (o resto da série fica acima de ~390 ppm): é o sensor desligando ou religando.
- Tratamento: removemos a duplicata, completamos a grade de 10 min, trocamos `-9999` **e as leituras vizinhas a cada bloco** por faltante, preenchemos por interpolação linear e marcamos como `imputado`. Amostras de treino e validação cujo **alvo** toca um ponto imputado são descartadas. O teste não tem pontos imputados.

**ETTh1**
- Todo dia 31 dos meses que têm 31 dias (14 dias) tem **todas as 7 colunas constantes por 24 h**: são dados preenchidos artificialmente, e não medições reais. 3 desses dias caem no teste.
- Tratamento: mantidos, para não quebrar a comparação com a literatura, mas marcados como `suspeito`. Toda avaliação reporta também `mse_sem_suspeitos`, que exclui as amostras cujo alvo toca esses dias.

## Interface dos modelos

Todos os modelos seguem a mesma interface, e a grade de experimentos é a mesma para todos:

```python
def criar(L, H):          # devolve um nn.Module
    return MeuModelo(L, H)
# forward recebe x com forma [lote, L, 1] e devolve [lote, H, 1]
ru.rodar_grade('NomeDoModelo', criar)
```
A normalização por janela (RevIN) é aplicada por fora, pelo `ru.ComRevIN`, igual para todos. Por isso nenhum modelo tem normalização própria.

## Equipe

Giovanna, João Pedro e Letícia.